### Evaluating GPT-5 model on aspect sentiment classification (ASC)

- 3 datasets were used for evaluation
- for each dataset, the evaluation pipeline was run 3 times using a different test split each time, namely with seed 42, 55 and 777. The test splits can be found in the dataset folder within this directory.
- The average score of each metric across the 3 runs was computed and the results are listed in the final report

To run, activate the `openai_environment` conda environment specified in the environements directory. Furthermore, replace the API_KEY placeholder with a real OpenAI API key.

In [ ]:
import json
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from openai import OpenAI

# instantiate api connection
client = OpenAI(
  api_key="API-KEY"
)

# load evaluation dataset
df = pd.read_csv("dataset/test.txt" , sep="\t")
pred_labels  = []

In [ ]:
# running inference from GPT-5 for each row in evaluation dataset
# tqdm enables progress of inference loop to be tracked
for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating GPT-5 on ASC"):
    prompt = f"""
    You are performing Aspect-Based Sentiment Analysis (ABSA).
    Given a sentence and a target aspect, classify the sentiment about that aspect.

    Sentiment label mapping:
    - 0 = negative
    - 1 = neutral
    - 2 = positive

    Return ONLY JSON following the provided schema.

    Sentence: {row['sentence']}
    Aspect: {row['aspect']}

    Respond with:
    - label: integer in [0,1,2]
    - rationale: short reason for the label (1-2 sentences)
    """.strip()

    resp = client.responses.create(
        model="gpt-5",
        input=prompt,                    
        store=False,
    )

    stripped_text = resp.output_text.strip()
    label_rationale_list = json.loads(stripped_text) 
    label_pred = label_rationale_list["label"]
    pred_labels.append(label_pred)

Evaluating OpenAI on ASC: 100%|██████████| 47/47 [02:39<00:00,  3.39s/it]


In [ ]:
# building output frame with pred_labels column
new_df = df.copy()
new_df["pred_label"] = pred_labels
y_true = new_df["label"].astype(int).values
y_pred = new_df["pred_label"].astype(int).values

# compute score
prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro",zero_division=0)
accuracy = accuracy_score(y_true, y_pred)
print(f"Accuracy:{accuracy}, Precision:{prec}, Recall:{rec}, F1-score:{f1}")

Accuracy:0.8297872340425532, Precision:0.5499999999999999, Recall:0.5666666666666668, F1-score:0.5580645161290323
